# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

from agents.deals import ScrapedDeal
from agents.deals_common import DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [4]:
import nest_asyncio
nest_asyncio.apply()
deals = ScrapedDeal.fetch(show_progress=True)

100%|███████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 5123.34it/s]


In [5]:
len(deals)

90

In [6]:
deals[44].describe()

"Title: DeWalt Daily Deals at Lowe's: Up to 50% off + free shipping\nDetails: As one of its daily deals, Lowe's has several power tool deals with discounts up to 50% off. We like the DeWalt 20V MAX Brushless Hammer Drill Combo Kit with two batteries and charger included for $229 ($100 off). Shop Now at Lowe's\nFeatures: \nURL: https://www.dealnews.com/De-Walt-Daily-Deals-at-Lowes-Up-to-50-off-free-shipping/21742536.html?iref=rss-c196"

In [7]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [8]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [9]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Jamo S 83 Center Channel Speaker for $49 + free shipping
Details: This same speaker will set you back $99 at Amazon. Buy Now at Adorama
Features: frequency response 65Hz to 26kHz Model: 1064330
URL: https://www.dealnews.com/products/Jamo/Jamo-S-83-Center-Channel-Speaker/490376.html?iref=rss-c142

Title: Certified Refurb JBL Wireless Microphone Set for $50 + free shi

In [10]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [11]:
result = get_recommendations()

In [12]:
print(type(result))
len(result.deals)

<class 'agents.deals_common.DealSelection'>


5

In [13]:
result.deals[1]

Deal(product_description='The Garmin Vivoactive 5 is a versatile fitness tracking smartwatch equipped with an AMOLED display for vibrant visuals. It features an impressive battery life of up to 11 days, ensuring you stay connected to your fitness goals without frequent charging. With 30 built-in indoor and GPS sports apps, along with sleep score and personalized coaching, this smartwatch is designed to elevate your fitness journey and help you monitor your health effectively.', price=219.0, url='https://www.dealnews.com/products/Garmin/Garmin-Vivoactive-5-Fitness-Tracking-Smart-Watch/490373.html?iref=rss-c142')

## ScannerAgent is using 'gpt-4o-mini'

In [14]:
from agents.scanner_agent import ScannerAgent

In [15]:
agent = ScannerAgent(show_progress=True)
result = agent.scan()

100%|█████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<?, ?it/s]


In [16]:
result.deals

[Deal(product_description="The Jamo S 83 Center Channel Speaker delivers impressive sound quality with a wide frequency response ranging from 65Hz to 26kHz. This speaker is designed to enhance your audio experience, whether you're watching movies or listening to music. With its sleek design and sturdy construction, it fits seamlessly into any home entertainment setup, providing clear dialogue and balanced sound. It's a great addition for audiophiles looking for high performance without breaking the bank.", price=49.0, url='https://www.dealnews.com/products/Jamo/Jamo-S-83-Center-Channel-Speaker/490376.html?iref=rss-c142'),
 Deal(product_description='The Garmin Vivoactive 5 is a fitness-focused smartwatch featuring an AMOLED display that ensures vibrant visuals. It comes equipped with a BodyBattery feature to monitor energy levels throughout the day, alongside 30 built-in sports apps suitable for both indoor and outdoor activities. With a battery life of up to 11 days, it also offers per

In [17]:
print(result.deals[0])

product_description="The Jamo S 83 Center Channel Speaker delivers impressive sound quality with a wide frequency response ranging from 65Hz to 26kHz. This speaker is designed to enhance your audio experience, whether you're watching movies or listening to music. With its sleek design and sturdy construction, it fits seamlessly into any home entertainment setup, providing clear dialogue and balanced sound. It's a great addition for audiophiles looking for high performance without breaking the bank." price=49.0 url='https://www.dealnews.com/products/Jamo/Jamo-S-83-Center-Channel-Speaker/490376.html?iref=rss-c142'


In [18]:
print(type(result))

<class 'agents.deals_common.DealSelection'>
